# Chapter 20 — Information Theory Basics
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Understand entropy, cross-entropy, and KL divergence
- Connect information theory to ML training objectives
- Apply mutual information for feature selection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.datasets import load_iris
from sklearn.feature_selection import mutual_info_classif

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Entropy and Information

**Shannon Entropy** measures the expected surprise (uncertainty):
$$H(X) = -\sum_{x} p(x)\log_2 p(x) \quad \text{(bits)}$$

Intuition:
- Fair coin: $H = -2 \times 0.5\log_2(0.5) = 1$ bit
- Biased coin p=0.9: $H ≈ 0.47$ bits (less surprising)
- Deterministic: $H = 0$ bits

**Cross-entropy** between true $p$ and model $q$:
$$H(p, q) = -\sum_x p(x)\log q(x) = H(p) + D_{KL}(p \| q)$$

**KL Divergence** (not symmetric!):
$$D_{KL}(p \| q) = \sum_x p(x)\log\frac{p(x)}{q(x)} \geq 0$$

In [ ]:
# Entropy as function of bias
p_values = np.linspace(0.001, 0.999, 300)
entropy = -p_values*np.log2(p_values) - (1-p_values)*np.log2(1-p_values)

plt.figure(figsize=(8, 4))
plt.plot(p_values, entropy, lw=2)
plt.scatter([0.5], [1.0], color="red", zorder=5, label="Max entropy at p=0.5 (1 bit)")
plt.xlabel("p (bias of coin)"); plt.ylabel("Entropy H(p) [bits]")
plt.title("Binary Entropy Function")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — KL Divergence

In [ ]:
def kl_divergence(p, q, eps=1e-10):
    p = np.array(p) + eps; q = np.array(q) + eps
    p, q = p/p.sum(), q/q.sum()
    return (p * np.log(p/q)).sum()

# Compare Gaussian distributions
mu_true, sig_true = 0, 1
for mu_q, sig_q in [(0, 1), (1, 1), (0, 2), (2, 2)]:
    p = stats.norm(mu_true, sig_true)
    q = stats.norm(mu_q, sig_q)
    x = np.linspace(-5, 7, 1000)
    dx = x[1]-x[0]
    kl = (p.pdf(x) * np.log((p.pdf(x)+1e-10)/(q.pdf(x)+1e-10)) * dx).sum()
    print(f"KL(N(0,1) || N({mu_q},{sig_q})) = {kl:.4f}")

In [ ]:
# Mutual Information for feature selection
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
mi_scores = mutual_info_classif(X_iris, y_iris, random_state=42)

plt.bar(iris.feature_names, mi_scores, color="steelblue")
plt.ylabel("Mutual Information (bits)")
plt.title("Mutual Information — Iris Feature Selection")
plt.tight_layout(); plt.show()
for name, score in zip(iris.feature_names, mi_scores):
    print(f"  {name}: {score:.4f} bits")

In [ ]:
# Cross-entropy loss in practice
def cross_entropy(y_true, y_pred, eps=1e-9):
    return -np.mean(y_true*np.log(y_pred+eps) + (1-y_true)*np.log(1-y_pred+eps))

probs = np.array([0.9, 0.8, 0.1, 0.3, 0.7])
labels = np.array([1,   1,   0,   0,   1])
ce = cross_entropy(labels, probs)
print(f"Cross-entropy loss: {ce:.4f}")

# Show how CE decreases as model improves
noise_levels = np.linspace(0, 0.45, 100)
ces = [cross_entropy(labels, np.clip(labels + rng.uniform(-n, n, len(labels)), 0.01, 0.99))
       for n in noise_levels]
plt.plot(noise_levels, ces)
plt.xlabel("Prediction noise"); plt.ylabel("Cross-entropy")
plt.title("Cross-Entropy vs. Prediction Quality")
plt.tight_layout(); plt.show()

In [ ]:
# Practice: entropy of uniform vs peaked distributions
distributions = [
    ("Uniform over 4", [0.25]*4),
    ("Peaked at 1", [0.7, 0.1, 0.1, 0.1]),
    ("Deterministic", [1.0, 0, 0, 0]),
    ("Two-peaked", [0.45, 0.45, 0.05, 0.05]),
]
for name, p in distributions:
    p_arr = np.array(p, dtype=float)
    h = -np.sum(p_arr[p_arr>0] * np.log2(p_arr[p_arr>0]))
    print(f"{name:<25} H={h:.4f} bits")